In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [3]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [4]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [5]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:
  
  #print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    #print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)

In [12]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [45]:
itos[X[1][2].item()]
for i in range (len(X)):
    ##for j in range(len(X[i])):
    print(itos[X[i][0].item()] + itos[X[i][1].item()] + itos[X[i][2].item()] + '  gives ' + itos[Y[i].item()])

...  gives e
..e  gives m
.em  gives m
emm  gives a
mma  gives .
...  gives o
..o  gives l
.ol  gives i
oli  gives v
liv  gives i
ivi  gives a
via  gives .
...  gives a
..a  gives v
.av  gives a
ava  gives .
...  gives i
..i  gives s
.is  gives a
isa  gives b
sab  gives e
abe  gives l
bel  gives l
ell  gives a
lla  gives .
...  gives s
..s  gives o
.so  gives p
sop  gives h
oph  gives i
phi  gives a
hia  gives .


In [7]:
C = torch.randn((27, 2))

In [8]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [9]:
a = C[1]
a.shape

torch.Size([2])

In [10]:
C = torch.randn((27, 2)) ## Here for our purposes we are taking each character of the dataset and representing it in 2 Dimensions


In [66]:
emb = C[X]
emb.shape ## 32 w

torch.Size([32, 3, 10])

In [47]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)

In [48]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [51]:
logits = h @ W2 + b2
logits.shape
counts = logits.exp()

In [52]:
prob = counts / counts.sum(1, keepdims=True)

In [53]:
loss = -prob[torch.arange(32), Y].log().mean()
loss

tensor(18.5906)

In [54]:
## The reason apart from learning is where instead of our own implementation of cross-entorpy loss using the pytorch is two fold
## Alot of intermediate values are there which take up space a
## main is when we calculate the expo of a very large positive number then it can come as infinity as it crosses the higher limit of the
## exponentiation operation. Pytorch.crossentropy kinda subtracts the largest number of the set from all and then performs exponentiation 
## on all negative numbers to ensure this does not happen
## also forward and backward passes are much more efficient 


In [61]:
logits = torch.tensor([-5,7,8,100])
counts = logits.exp()
prob = counts/counts.sum() 
print(counts)
prob

tensor([6.7379e-03, 1.0966e+03, 2.9810e+03,        inf])


tensor([0., 0., 0., nan])

In [71]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [72]:
for p in parameters:
  p.requires_grad = True

In [77]:
for i in range(100):
  
  # forward pass
  emb = C[X]
  h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
  logits = h @ W2 + b2 # (32, 27)
  loss = F.cross_entropy(logits, Y)
  print(f"Loss in the {i+1} round is {loss}")
  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
   #updatr
  for p in  parameters:
      p.data += -0.1 * p.grad
  


Loss in the 1 round is 97.86323547363281
Loss in the 2 round is 88.3282241821289
Loss in the 3 round is 79.59536743164062
Loss in the 4 round is 70.95570373535156
Loss in the 5 round is 60.914119720458984
Loss in the 6 round is 49.95695495605469
Loss in the 7 round is 39.279361724853516
Loss in the 8 round is 28.568456649780273
Loss in the 9 round is 21.04163360595703
Loss in the 10 round is 16.920408248901367
Loss in the 11 round is 14.167389869689941
Loss in the 12 round is 11.863746643066406
Loss in the 13 round is 10.081355094909668
Loss in the 14 round is 8.78769302368164
Loss in the 15 round is 7.817413806915283
Loss in the 16 round is 7.002355098724365
Loss in the 17 round is 6.282127857208252
Loss in the 18 round is 5.66121768951416
Loss in the 19 round is 5.140425682067871
Loss in the 20 round is 4.698785305023193
Loss in the 21 round is 4.315305233001709
Loss in the 22 round is 3.979773998260498
Loss in the 23 round is 3.6790640354156494
Loss in the 24 round is 3.401268243789